# audiobooks-generation: демонстрація в Colab

Короткий огляд усього, що вміє репозиторій [flonado/audiobooks-generation](https://github.com/flonado/audiobooks-generation): книга → озвучення → `.m4b`. Ноутбук **самодостатній**: нічого завантажувати з комп'ютера не треба. Він сам клонує репозиторій і сам качає потрібні моделі та голоси з Hugging Face.

**Перед запуском:** Runtime → Change runtime type → **GPU** (T4 підійде). Розділи 0 і 1 виконуй по черзі, далі кожен розділ майже незалежний (розділи 3–4 потребують розділів 0–2).

| Розділ | Що там | Час |
|---|---|---|
| 0 | Встановлення (два окремі середовища, бо Higgs і StyleTTS2 мають несумісні версії `transformers`) | 5–10 хв, раз |
| 1 | Огляд репозиторію і команд | секунди |
| 2 | Книга → розділи, підготовка тексту | секунди |
| 3 | **StyleTTS2**: список голосів, озвучення, порівняння голосів, наголоси, швидкість | ≈5 хв |
| 4 | **Higgs 3**: клонування голосу за зразком (без референсу, зі зразка StyleTTS2, зі свого запису) | ≈15 хв (ваги ≈9 ГБ) |
| 5 | Збирання `.m4b` з глав | секунди |

Переклад через Lapa сюди навмисно не входить: йому потрібен окремий сервер llama.cpp.

**Ліцензії.** Ваги Higgs 3 мають некомерційну ліцензію; клонувати чужий голос можна лише з дозволу мовця. Голоси StyleTTS2 мають умови своєї моделі (`patriotyk/styletts2_ukrainian_multispeaker`).

## 0. Встановлення

In [ ]:
import os, sys, subprocess, json, glob, shutil, time
ROOT = "/content/audiobooks-generation"
REPO = "https://github.com/flonado/audiobooks-generation.git"

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "GPU не знайдено: увімкни його в Runtime → Change runtime type")
if os.path.exists(ROOT):
    !git -C {ROOT} pull -q
else:
    !git clone -q {REPO} {ROOT}
!git -C {ROOT} log --oneline -1

In [ ]:
# Два окремих середовища (venv), які бачать системний torch з CUDA, але мають свої версії transformers:
#   styletts2: transformers==5.0.0 (+ ukrainian-word-stress, ipa-uk, styletts2-inference)
#   higgs3:    transformers>=5.5   (клас HiggsAudioV2TokenizerModel є лише в нових версіях)
!pip -q install uv
ENVS = {"styletts2": "/content/venv-styletts2", "higgs3": "/content/venv-higgs"}
for path in ENVS.values():
    if not os.path.exists(path):
        !uv venv -q --system-site-packages --python {sys.executable} {path}
PY = {k: f"{v}/bin/python" for k, v in ENVS.items()}

!uv pip install -q --python {PY["styletts2"]} "transformers==5.0.0" accelerate einops einops_exts munch sentencepiece ctranslate2 soundfile librosa scipy stanza pyyaml "git+https://github.com/patriotyk/ukrainian-word-stress.git" "git+https://github.com/patriotyk/ipa-uk.git" "git+https://github.com/patriotyk/styletts2-inference@209352700cf703a05ff94501290095dd9072fd5f"
!uv pip install -q --python {PY["higgs3"]} "transformers>=5.5" accelerate tokenizers soundfile pyyaml
print("готово")

Далі йде невеликий «виконавець» `worker.py`: він запускається у потрібному середовищі, завантажує рушій **один раз** і озвучує список фраз. Функція `run_tts` нижче викликає його й показує плеєри.

In [ ]:
%%writefile /content/worker.py
import sys, json, time
sys.path.insert(0, "/content/audiobooks-generation/src")
from pathlib import Path
from audiobooks.audio import save_wav
from audiobooks.tts.local import make_engine, synthesize_text

job = json.load(open(sys.argv[1], encoding="utf-8"))
engine = job["engine"]
t0 = time.time()
eng = make_engine(engine, job["voice"], job.get("speed", 1.0), **job.get("tuning", {}))
print(f"рушій {engine} завантажено за {time.time() - t0:.0f} с", flush=True)

def switch_voice(voice):
    if engine == "higgs3":
        eng.set_voice(voice)
    else:  # StyleTTS2: голос = стиль 1x256, модель перезавантажувати не треба
        from audiobooks.tts.styletts2_engine import VOICES_DIR
        eng.style = eng.torch.load(VOICES_DIR / f"{voice}.pt").to(job["tuning"]["device"])
        eng.dur_style = None

current = job["voice"]
for item in job["items"]:
    if item.get("voice", current) != current:
        current = item["voice"]
        switch_voice(current)
    eng.speed = item.get("speed", job.get("speed", 1.0))
    if engine == "higgs3":
        eng.tags = item.get("tags", "")
    print(f"\n--- {item['out']}  (голос: {current})", flush=True)
    t = time.time()
    pcm = synthesize_text(eng, item["text"], label="речення")
    save_wav(Path(item["out"]), pcm, eng.sample_rate)
    sec, took = len(pcm) / 2 / eng.sample_rate, time.time() - t
    print(f"  {sec:.1f} с звуку за {took:.1f} с  (швидше за реальний час у {sec / took:.1f}x)", flush=True)

In [ ]:
from IPython.display import Audio, display

def run_tts(engine, voice, items, tuning=None, speed=1.0):
    """items: [{"text": ..., "out": ..., "voice": (необов'язково), "speed": ..., "tags": ...}]. Показує плеєри."""
    tuning = {"device": "cuda", **(tuning or {})}
    job = {"engine": engine, "voice": voice, "speed": speed, "tuning": tuning, "items": items}
    json.dump(job, open("/content/job.json", "w", encoding="utf-8"), ensure_ascii=False)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen([PY[engine], "/content/worker.py", "/content/job.json"], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, env=env)
    while chunk := os.read(p.stdout.fileno(), 4096):
        sys.stdout.write(chunk.decode("utf-8", "replace")); sys.stdout.flush()
    if p.wait():
        raise RuntimeError(f"worker завершився з кодом {p.returncode}")
    for it in items:
        print(os.path.basename(it["out"]), "|", it.get("voice", voice))
        display(Audio(filename=it["out"]))

## 1. Огляд репозиторію

Усе керується однією командою `audiobook` (`src/audiobooks/cli.py`). Кроки:
`chapters` (розділи книги) → `translate` (Lapa) → `gemini` / `local` (озвучення) → `m4b` (збирання).

In [ ]:
!ls {ROOT} {ROOT}/src/audiobooks {ROOT}/src/audiobooks/tts {ROOT}/tools

In [ ]:
AB = f"PYTHONPATH={ROOT}/src {PY['styletts2']} -m audiobooks.cli"
!{AB} --help

## 2. Книга → розділи → текст для озвучення

Читачі форматів: `fb2`, `epub`, `pdf`, а також папка з перекладом (`NN_назва.txt`: перший рядок назва розділу, далі абзаци). Для демо створимо маленьку «книгу» з двох розділів.

In [ ]:
BOOK = "/content/demo_book"
os.makedirs(BOOK, exist_ok=True)
open(f"{BOOK}/01_Ранок.txt", "w", encoding="utf-8").write(
    "Ранок\n"
    "Над лісом повільно підіймався туман. Птахи ще мовчали, і лише струмок дзюркотів між камінням.\n"
    "«Сьогодні буде гарний день», — сказала вона. Було рівно 7 годин ранку.\n")
open(f"{BOOK}/02_Дорога.txt", "w", encoding="utf-8").write(
    "Дорога\n"
    "Вони йшли цілий день і зупинилися лише біля старого мосту. Попереду було ще 15 кілометрів.\n")
!{AB} chapters {BOOK}

In [ ]:
# Що відбувається з текстом перед озвученням: прибирання, числа словами, розбиття на речення
sys.path.insert(0, f"{ROOT}/src")
from audiobooks.books import load_book
from audiobooks.text import prepare_text, spell_numbers, split_sentences
title, body = load_book(BOOK)[0]
text = spell_numbers(prepare_text(title, body))
print(text, "\n")
for i, s in enumerate(split_sentences(text, 120), 1):
    print(i, s)

## 3. StyleTTS2: швидкий український TTS зі списком голосів

Модель `patriotyk/styletts2_ukrainian_multispeaker`. Голос = крихітний файл-стиль (`.pt`, ≈2 КБ), тож перемикання голосів миттєве. Стилі беремо з демо-простору автора моделі на Hugging Face. Наголоси розставляє `ukrainian-word-stress`; вручну їх можна задати знаком `+` перед голосною (`замо+к`) або в `configs/stress*.yaml`.

In [ ]:
from huggingface_hub import snapshot_download
VOICES = f"{ROOT}/data/models/styletts2/voices"
os.makedirs(VOICES, exist_ok=True)
tmp = snapshot_download("patriotyk/styletts2-ukrainian", repo_type="space", allow_patterns=["voices/*.pt"],
                        local_dir="/content/st2_space")
for f in glob.glob(f"{tmp}/voices/*.pt"):
    shutil.copy(f, VOICES)
ST2_VOICES = sorted(os.path.splitext(f)[0] for f in os.listdir(VOICES) if f.endswith(".pt") and not f.endswith(".dur.pt"))
print(len(ST2_VOICES), "голосів:")
print("\n".join(ST2_VOICES))

In [ ]:
#@title Озвучення одним голосом
VOICE = "Марічка Штирбулова"  #@param {type:"string"}
TEXT = "Над лісом повільно підіймався туман. Птахи ще мовчали, і лише струмок дзюркотів між камінням."  #@param {type:"string"}
SPEED = 1.0  #@param {type:"number"}
os.makedirs("/content/out", exist_ok=True)
run_tts("styletts2", VOICE, [{"text": TEXT, "out": "/content/out/st2_one.wav"}], speed=SPEED)

In [ ]:
# Одна фраза різними голосами (модель вантажиться раз, голоси перемикаються миттєво)
COMPARE = ["Марічка Штирбулова", "Тетяна Гончарова", "Тарас Василюк", "Роман Куліш"]
PHRASE = "Він зупинився біля старого мосту й довго дивився на воду."
run_tts("styletts2", COMPARE[0], [{"text": PHRASE, "voice": v, "out": f"/content/out/st2_cmp_{i}.wav"}
                                  for i, v in enumerate(COMPARE)])

In [ ]:
# Наголоси: «+» перед голосною задає наголос вручну; швидкість: 0.85 повільніше, 1.15 швидше
items = [
    {"text": "Цей за+мок стоїть на горі, а цей замо+к висить на дверях.", "out": "/content/out/st2_stress.wav"},
    {"text": PHRASE, "speed": 0.85, "out": "/content/out/st2_slow.wav"},
    {"text": PHRASE, "speed": 1.15, "out": "/content/out/st2_fast.wav"},
]
run_tts("styletts2", "Марічка Штирбулова", items)

Те саме через справжню команду репозиторію, тобто озвучення розділу книги (`audiobook local`):

In [ ]:
!{AB} local {BOOK} -c 1 --engine styletts2 --voice "Марічка Штирбулова" --device cuda --out-dir /content/out/cli
display(Audio(filename=glob.glob("/content/out/cli/*.wav")[0]))

## 4. Higgs 3: клонування голосу за зразком

`bosonai/higgs-tts-3-4b` (~4 млрд параметрів): голос задається **референсним записом (6–14 с) і його точним текстом**, без жодного навчання. Модель продовжує читати текст тим самим голосом. На T4 це приблизно у 2 рази швидше за реальний час, а ваги важать ≈9 ГБ.

Ліцензія ваг некомерційна; клонувати можна лише голос, на який є дозвіл мовця.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
H = f"{ROOT}/data/models/higgs3"
snapshot_download("bosonai/higgs-tts-3-4b", local_dir=f"{H}/higgs-tts-3-4b",
                  allow_patterns=["*.json", "*.safetensors", "*.jinja", "LICENSE"])
shutil.copy(hf_hub_download("bosonai/higgs-audio-v2-tokenizer", "config.json"), f"{H}/higgs_audio_v2_tokenizer_config.json")
os.makedirs(f"{H}/voices", exist_ok=True)
print(os.listdir(f"{H}/higgs-tts-3-4b"))

### 4.1 Без референсу та зі зразка StyleTTS2

Як референс візьмемо **штучний** запис: StyleTTS2 читає короткий текст обраним голосом, і цей запис стає зразком для Higgs. Так клонування можна спробувати без чужих записів.

In [ ]:
#@title Референс зі StyleTTS2
REF_VOICE = "Тетяна Гончарова"  #@param {type:"string"}
REF_TEXT = "Ранок видався холодним і ясним, а над лісом повільно підіймався густий білий туман."  #@param {type:"string"}
name = "st2_" + REF_VOICE.replace(" ", "_")
run_tts("styletts2", REF_VOICE, [{"text": REF_TEXT, "out": f"{H}/voices/{name}.wav"}])
open(f"{H}/voices/{name}.txt", "w", encoding="utf-8").write(REF_TEXT)
print("голос Higgs створено:", name)

In [ ]:
#@title Higgs: власний голос моделі і клон
TEXT_H = "Вони йшли цілий день і зупинилися лише біля старого мосту. Попереду було ще багато кілометрів дороги."  #@param {type:"string"}
SEED = 1  #@param {type:"integer"}
TEMPERATURE = 0.8  #@param {type:"number"}
items = [
    {"text": TEXT_H, "voice": "(без референсу)", "out": "/content/out/higgs_noref.wav"},
    {"text": TEXT_H, "voice": name, "out": "/content/out/higgs_clone_st2.wav"},
]
run_tts("higgs3", "(без референсу)", items, tuning={"seed": SEED, "temperature": TEMPERATURE, "dtype": "bf16"})

### 4.2 Клонування зі свого запису

Запиши (або знайди) 6–14 секунд чистого мовлення **українською, без музики** і впиши в `REF_TEXT_OWN` те, що там сказано, **дослівно**: від цього сильно залежить схожість. Формат: wav, mp3 чи m4a. Лише свій голос або голос людини, яка дозволила.

In [ ]:
#@title Завантаж свій зразок
from google.colab import files
REF_TEXT_OWN = "Впиши сюди дослівний текст запису."  #@param {type:"string"}
OWN_NAME = "my_voice"  #@param {type:"string"}
up = files.upload()
src = next(iter(up))
!ffmpeg -y -loglevel error -i "{src}" -ar 24000 -ac 1 -t 14 "{H}/voices/{OWN_NAME}.wav"
open(f"{H}/voices/{OWN_NAME}.txt", "w", encoding="utf-8").write(REF_TEXT_OWN)
print("зразок збережено (моно, 24 кГц, не довше 14 с):")
display(Audio(filename=f"{H}/voices/{OWN_NAME}.wav"))

In [ ]:
#@title Озвуч будь-який текст своїм голосом (і з тегами стилю)
TEXT_OWN = "Привіт! Це моє клоноване озвучення, зроблене моделлю Higgs три."  #@param {type:"string"}
items = [
    {"text": TEXT_OWN, "out": "/content/out/higgs_own.wav"},
    {"text": TEXT_OWN, "tags": "<|emotion:contentment|>", "out": "/content/out/higgs_own_tag.wav"},
    {"text": TEXT_OWN, "speed": 1.2, "out": "/content/out/higgs_own_fast.wav"},
]
run_tts("higgs3", OWN_NAME, items, tuning={"seed": SEED, "temperature": TEMPERATURE, "dtype": "bf16"})

## 5. Збирання `.m4b` з розділів

`audiobook m4b <config.yaml>` склеює wav-розділи в один файл із главами (AAC). Тут збираємо з того, що озвучили вище; у реальному проєкті конфіг лежить у `configs/` (приклад: `configs/audiobook.yaml`).

In [ ]:
import yaml
chapters = [
    {"file": "/content/out/st2_one.wav", "title": "StyleTTS2: один голос"},
    {"file": "/content/out/st2_stress.wav", "title": "StyleTTS2: наголоси"},
    {"file": "/content/out/higgs_clone_st2.wav", "title": "Higgs: клон"},
]
chapters = [c for c in chapters if os.path.exists(c["file"])]
cfg = {"output": "/content/demo.m4b", "title": "Демо", "author": "audiobooks-generation", "bitrate": "64k", "chapters": chapters}
yaml.safe_dump(cfg, open("/content/demo.yaml", "w", encoding="utf-8"), allow_unicode=True)
!{AB} m4b /content/demo.yaml

In [ ]:
# за бажанням: забрати результат на свій комп'ютер
from google.colab import files
files.download("/content/demo.m4b")